In [ ]:
!pip install -q transformers torch pandas plotly gradio


In [2]:
import pandas as pd
import plotly.express as px
import gradio as gr
import torch

from transformers import pipeline
from pathlib import Path
from google.colab import files
from datetime import datetime

print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__)

All libraries imported successfully!
PyTorch version: 2.11.0+cpu


In [3]:
MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

device = 0 if torch.cuda.is_available() else -1

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    device=device
)

print("Sentiment analysis model loaded successfully!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentiment analysis model loaded successfully!


In [4]:
def analyze_sentiment(text):
    if not isinstance(text, str) or not text.strip():
        return {
            "text": text if isinstance(text, str) else "",
            "sentiment": "Unknown",
            "confidence": 0.0
        }

    result = sentiment_pipeline(
        text[:512],
        truncation=True
    )[0]

    model_label = result["label"]
    confidence = float(result["score"])

    # Use a conservative rule for uncertain predictions.
    if confidence < 0.60:
        sentiment = "Neutral"
    elif model_label == "POSITIVE":
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return {
        "text": text,
        "sentiment": sentiment,
        "confidence": round(confidence * 100, 2)
    }


print(analyze_sentiment("This product is amazing!"))
print(analyze_sentiment("The product is terrible and disappointing."))

{'text': 'This product is amazing!', 'sentiment': 'Positive', 'confidence': 99.99}
{'text': 'The product is terrible and disappointing.', 'sentiment': 'Negative', 'confidence': 99.98}


In [5]:
test_reviews = [
    "I absolutely love this product. It works perfectly!",
    "This is the worst purchase I have ever made.",
    "The delivery arrived yesterday.",
    "Excellent quality and fast delivery.",
    "Very poor quality. I am disappointed."
]

for review in test_reviews:
    result = analyze_sentiment(review)

    print("Review:", result["text"])
    print("Sentiment:", result["sentiment"])
    print("Confidence:", str(result["confidence"]) + "%")
    print("-" * 50)

Review: I absolutely love this product. It works perfectly!
Sentiment: Positive
Confidence: 99.99%
--------------------------------------------------
Review: This is the worst purchase I have ever made.
Sentiment: Negative
Confidence: 99.98%
--------------------------------------------------
Review: The delivery arrived yesterday.
Sentiment: Positive
Confidence: 97.24%
--------------------------------------------------
Review: Excellent quality and fast delivery.
Sentiment: Positive
Confidence: 99.99%
--------------------------------------------------
Review: Very poor quality. I am disappointed.
Sentiment: Negative
Confidence: 99.98%
--------------------------------------------------


In [6]:
reviews_data = {
    "review_id": list(range(1, 16)),
    "review": [
        "Amazing product, excellent quality!",
        "I love this product and would buy it again.",
        "The quality is fantastic.",
        "Very useful and easy to use.",
        "Excellent value for money.",
        "Terrible product. It stopped working.",
        "Very disappointed with this purchase.",
        "Poor quality and bad packaging.",
        "I regret buying this product.",
        "The product is not worth the price.",
        "The package arrived yesterday.",
        "The box contains two items.",
        "The product is available in three colors.",
        "Delivery took two days.",
        "The item matches the description."
    ]
}

reviews_df = pd.DataFrame(reviews_data)

reviews_df.head()

,review_id,review
0,1,"Amazing product, excellent quality!"
1,2,I love this product and would buy it again.
2,3,The quality is fantastic.
3,4,Very useful and easy to use.
4,5,Excellent value for money.


In [7]:
results = []

for review in reviews_df["review"]:
    results.append(analyze_sentiment(review))

reviews_df["sentiment"] = [
    result["sentiment"] for result in results
]

reviews_df["confidence"] = [
    result["confidence"] for result in results
]

reviews_df["analyzed_at"] = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

reviews_df.head(15)

,review_id,review,sentiment,confidence,analyzed_at
0,1,"Amazing product, excellent quality!",Positive,99.99,2026-10-27 11:35:33
1,2,I love this product and would buy it again.,Positive,99.98,2026-10-27 11:35:33
2,3,The quality is fantastic.,Positive,99.99,2026-10-27 11:35:33
3,4,Very useful and easy to use.,Positive,99.98,2026-10-27 11:35:33
4,5,Excellent value for money.,Positive,99.99,2026-10-27 11:35:33
5,6,Terrible product. It stopped working.,Negative,99.98,2026-10-27 11:35:33
6,7,Very disappointed with this purchase.,Negative,99.98,2026-10-27 11:35:33
7,8,Poor quality and bad packaging.,Negative,99.98,2026-10-27 11:35:33
8,9,I regret buying this product.,Negative,99.83,2026-10-27 11:35:33
9,10,The product is not worth the price.,Negative,99.98,2026-10-27 11:35:33


In [8]:
sentiment_summary = (
    reviews_df["sentiment"]
    .value_counts()
    .reset_index()
)

sentiment_summary.columns = ["Sentiment", "Count"]

sentiment_summary

,Sentiment,Count
0,Positive,9
1,Negative,6


In [9]:
fig = px.bar(
    sentiment_summary,
    x="Sentiment",
    y="Count",
    color="Sentiment",
    title="Product Review Sentiment Distribution",
    labels={
        "Sentiment": "Predicted Sentiment",
        "Count": "Number of Reviews"
    },
    text="Count"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [10]:
fig_pie = px.pie(
    sentiment_summary,
    names="Sentiment",
    values="Count",
    title="Percentage of Reviews by Sentiment",
    hole=0.35
)

fig_pie.update_layout(title_x=0.5)

fig_pie.show()

In [11]:
def analyze_single_review(review):
    result = analyze_sentiment(review)

    return (
        f"### Predicted Sentiment: {result['sentiment']}\n\n"
        f"**Confidence score:** {result['confidence']}%\n\n"
        f"**Review:** {result['text']}"
    )


with gr.Blocks(title="AI-Based Sentiment Analyzer") as single_app:

    gr.Markdown(
        """
        # AI-Based Sentiment Analyzer

        Enter an English product review or social media comment
        to analyze its sentiment.
        """
    )

    review_input = gr.Textbox(
        label="Enter your review",
        placeholder="Example: This product is excellent!",
        lines=4
    )

    analyze_button = gr.Button("Analyze Sentiment")

    result_output = gr.Markdown()

    analyze_button.click(
        fn=analyze_single_review,
        inputs=review_input,
        outputs=result_output
    )

print("Single-review interface created successfully!")

Single-review interface created successfully!


In [12]:
single_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec8440316218fecd54.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
def analyze_csv(file_path):
    if file_path is None:
        return None, "Please upload a CSV file."

    try:
        df = pd.read_csv(file_path)

        if "review" not in df.columns:
            return None, (
                "The uploaded CSV must contain a column named 'review'."
            )

        df["review"] = df["review"].fillna("").astype(str)

        results = [
            analyze_sentiment(review)
            for review in df["review"]
        ]

        df["sentiment"] = [
            result["sentiment"] for result in results
        ]

        df["confidence"] = [
            result["confidence"] for result in results
        ]

        output_path = "/content/sentiment_analysis_results.csv"

        df.to_csv(output_path, index=False)

        return output_path, "Analysis completed successfully!"

    except Exception as error:
        return None, f"Error: {error}"

In [14]:
with gr.Blocks(title="Batch Sentiment Analyzer") as batch_app:

    gr.Markdown(
        """
        # Batch Sentiment Analysis

        Upload a CSV file with a column named `review`.
        The application will classify each review and export the results.
        """
    )

    csv_input = gr.File(
        label="Upload Reviews CSV",
        file_types=[".csv"],
        type="filepath"
    )

    batch_button = gr.Button("Analyze All Reviews")

    download_output = gr.File(label="Download Analysis Results")

    status_output = gr.Textbox(label="Status")

    batch_button.click(
        fn=analyze_csv,
        inputs=csv_input,
        outputs=[download_output, status_output]
    )

print("Batch analysis interface created successfully!")

Batch analysis interface created successfully!


In [15]:
batch_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f53cba4d620b1498e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
sample_csv = pd.DataFrame({
    "review": [
        "Excellent product and great quality!",
        "I hate this product.",
        "The delivery arrived today.",
        "Very useful and easy to operate.",
        "Poor quality and disappointing service."
    ]
})

sample_csv.to_csv(
    "/content/sample_reviews.csv",
    index=False
)

print("Sample CSV created successfully!")

Sample CSV created successfully!


In [17]:
files.download("/content/sample_reviews.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
report_path = "/content/sentiment_analysis_report.csv"

reviews_df.to_csv(report_path, index=False)

print("Report saved successfully:", report_path)

files.download(report_path)

Report saved successfully: /content/sentiment_analysis_report.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
chart_path = "/content/sentiment_dashboard.html"

fig.write_html(chart_path)

print("Interactive chart saved successfully!")

files.download(chart_path)

Interactive chart saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>